In [1]:
from google import genai
from google.genai.types import EmbedContentConfig
from google.genai import types
import pandas as pd
import time
from google.cloud import bigquery
from google.cloud.exceptions import NotFound

In [2]:
client = genai.Client(vertexai=True, project="resume-radar-thuan", location="us-central1")

In [3]:
bq = bigquery.Client(project="resume-radar-thuan")

In [4]:
df = pd.read_csv('data/postings.csv')
df = df.dropna(subset = ['formatted_experience_level'])
df['text'] = df.title +  ' ' + df.description.fillna("")
X = df.text
y = df.formatted_experience_level
df['text'].str.len().mean()

np.float64(3910.1427996611606)

In [5]:
df_sample = df.sample(n=10000, random_state = 42)

In [6]:
try:
    done = {row.job_id for row in bq.query("SELECT job_id FROM resume_radar.posting_embeddings").result()}
except NotFound:
    done = set()

In [7]:
batch_size = 100

try:
    done = {row.job_id for row in bq.query("SELECT job_id FROM resume_radar.posting_embeddings").result()}
except NotFound:
    done = set()
todo = df_sample[~df_sample["job_id"].isin(done)]
for start in range(0, len(todo), batch_size):
    batch = todo.iloc[start:start +batch_size]
    
    for attempt in range(5):
        try:
            embed_content = client.models.embed_content(
                model="gemini-embedding-001",
                contents=batch.text.to_list(),
                config=types.EmbedContentConfig(output_dimensionality=768),)
            break
        except Exception as e:
            print(f"batch {start}: attempt {attempt} failed: {e}")
            time.sleep(2 ** attempt)
    else:
        raise RuntimeError(f"batch {batch} failed 5 times")
    
    rows = pd.DataFrame({
    "job_id": batch["job_id"].to_list(),
    "embedding": [e.values for e in embed_content.embeddings],
    "model": "gemini-embedding-001-768",
    })
    bq.load_table_from_dataframe(rows, "resume_radar.posting_embeddings").result()
    print(f"done {start + len(batch)} / {len(todo)}")



In [23]:
query = """
WITH target AS (
  SELECT embedding FROM resume_radar.posting_embeddings WHERE job_id = 3905341299
)
SELECT
  p.title,
  p.job_id,
  ML.DISTANCE(target.embedding, o.embedding, 'COSINE') AS cos_dist
FROM resume_radar.posting_embeddings AS o
CROSS JOIN target
JOIN resume_radar.postings AS p ON p.job_id = o.job_id
WHERE o.job_id != 3905341299
ORDER BY cos_dist ASC      
LIMIT 10
"""
for row in bq.query(query).result():
    print(row)

Row(('Web Designer I', 3904402111, 0.21990431061940063), {'title': 0, 'job_id': 1, 'cos_dist': 2})
Row(('Data Scientist', 3904363560, 0.23739560201518262), {'title': 0, 'job_id': 1, 'cos_dist': 2})
Row(('Senior Data Scientist, Experimentation & Personalization', 3894285749, 0.2495690208598913), {'title': 0, 'job_id': 1, 'cos_dist': 2})
Row(('Data Scientist I', 3904960773, 0.26596970265235476), {'title': 0, 'job_id': 1, 'cos_dist': 2})
Row(('Data Analytics Consultant', 3900089227, 0.2690381581809197), {'title': 0, 'job_id': 1, 'cos_dist': 2})
Row(('Data Scientist Lead – Telematics (Remote)', 3894627680, 0.27370603422463313), {'title': 0, 'job_id': 1, 'cos_dist': 2})
Row(('Data Scientist', 3906226694, 0.2812813937142221), {'title': 0, 'job_id': 1, 'cos_dist': 2})
Row(('Data Scientist III, Innovation', 3905885501, 0.2866950120680073), {'title': 0, 'job_id': 1, 'cos_dist': 2})
Row(('Associate Predictive Modeler II', 3885828791, 0.2899849299257812), {'title': 0, 'job_id': 1, 'cos_dist': 2})

In [18]:
query = """
SELECT job_id, title FROM resume_radar.postings 
WHERE job_id IN (SELECT job_id FROM resume_radar.posting_embeddings)
AND LOWER(title) LIKE '%data scientist%' LIMIT 5
"""
for row in bq.query(query).result():
    print(row)

Row((3905341299, 'Data Scientist'), {'job_id': 0, 'title': 1})
Row((3904960773, 'Data Scientist I'), {'job_id': 0, 'title': 1})
Row((3885812100, 'junior java programmer/data scientist'), {'job_id': 0, 'title': 1})
Row((3905885501, 'Data Scientist III, Innovation'), {'job_id': 0, 'title': 1})
Row((3902834352, 'Vice President/P&C Actuary/Data Scientist - PR12682'), {'job_id': 0, 'title': 1})


In [24]:
query = """
SELECT job_id, title, description FROM resume_radar.postings 
WHERE job_id IN (SELECT job_id FROM resume_radar.posting_embeddings)
AND job_id = 3905341299 or job_id = 3904402111
"""
for row in bq.query(query).result():
    print(row)

Row((3905341299, 'Data Scientist', "Overview\n\nProvide analytical insights by extracting data through machine learning, programming, data modeling, and advanced mathematics to recognize complex patterns and identify opportunities to drive impact across the organization. Developing an understanding of business needs and objectives through the use of basic descriptive, predictive, and prescriptive models. Perform routine assignments with increasing scope and complexity. Work under close supervision with some latitude for problem solving within existing data platforms and tools. Developing professional with basic skill set and proficiency.\n\nResponsibilities\n\nDesign, develop, and evaluate basic/routine predictive models and algorithms with some complexityAnalyze and interpret results with some complexityLimited judgment and discretion within defined procedures and practicesDevelop and code basic software programs, algorithms, and automated processesUse modeling and trend analysis to a